# 01 · SQL Detective — מתי JOIN מכפיל רשומות?
**Goal:** חקור נתונים, זהה מפתח עסקי, וראה Query Plan.  
**Setup:** Python + SQLite stdlib (ללא שרת), Matplotlib אופציונלי.  
**CS bridge:** relational algebra, B-Trees, query planners.  
**הבדל חשוב:** `EXPLAIN QUERY PLAN` של SQLite *אינו* `EXPLAIN ANALYZE` של PostgreSQL; בפרויקט האמיתי נבחן את Postgres בנפרד.

## צפה ולמד — Harvard CS50 SQL

[**פתח את הסרטון המקורי ב־YouTube**](https://www.youtube.com/watch?v=WXk7yDqsKxs) · מקטע מומלץ: 5–15 דקות לנושא שבו זיהית פער.

[![Video thumbnail](https://img.youtube.com/vi/WXk7yDqsKxs/hqdefault.jpg)](https://www.youtube.com/watch?v=WXk7yDqsKxs)

ב־Jupyter/VS Code הנגן בתא הבא עשוי לפעול אם יש רשת; ב־GitHub preview הנגן אינו מובטח. תוכל להמשיך בתרגיל גם בלי וידאו.

In [1]:
# Optional video player (does not download or store the video)
try:
    from IPython.display import display, YouTubeVideo
    display(YouTubeVideo("WXk7yDqsKxs", width=720, height=405))
except ImportError:
    print("Open the YouTube link in the Markdown above.")

**וידאו משלים — CS50 SQL:** [צפו בסרטון המקורי](https://www.youtube.com/watch?v=WXk7yDqsKxs) (נפתח באתר YouTube). ב־GitHub Preview הווידאו עלול לא להיטען; ההסברים והתרגילים כאן עובדים גם בלעדיו.

## 1. מינימום נתונים, שתי טבלאות
קח שתי דקות לנבא מה יקרה אם לכל אירוע יהיו שתי תגיות. למה `COUNT(*)` אחרי JOIN עלול לשקר?

In [2]:
import sqlite3
conn=sqlite3.connect(":memory:")
conn.executescript("""
CREATE TABLE events(event_id TEXT PRIMARY KEY, region TEXT NOT NULL, magnitude REAL NOT NULL);
CREATE TABLE tags(event_id TEXT NOT NULL, label TEXT NOT NULL);
INSERT INTO events VALUES ('e1','North',2.5),('e2','South',4.0),('e3','North',3.1);
INSERT INTO tags VALUES ('e1','reviewed'),('e1','priority'),('e2','priority');
""")
print(conn.execute("SELECT COUNT(*) FROM events").fetchone()[0])

3


## 2. כשל קלאסי: cardinality
כמה rows תחזיר שאילתת ה־JOIN? כמה אירועים שונים קיימים באמת?

In [3]:
rows=conn.execute("SELECT e.event_id,t.label FROM events e LEFT JOIN tags t ON t.event_id=e.event_id ORDER BY e.event_id").fetchall()
print(rows)
count_joined = len(rows)
unique_events = conn.execute("SELECT COUNT(DISTINCT event_id) FROM events").fetchone()[0]
print("joined rows:",count_joined, "actual events:",unique_events)
assert count_joined == 4 and unique_events == 3

[('e1', 'priority'), ('e1', 'reviewed'), ('e2', 'priority'), ('e3', None)]
joined rows: 4 actual events: 3


## 3. Query Plan — לפני ואחרי אינדקס
האם planner בחר index? בדוק `EXPLAIN QUERY PLAN`, אך אל תניח ש־index תמיד מהיר יותר; בטבלה קטנה overhead יכול להיות גדול מהתועלת.

In [4]:
query="SELECT * FROM events WHERE region = 'North'"
print("before:",conn.execute("EXPLAIN QUERY PLAN "+query).fetchall())
conn.execute("CREATE INDEX idx_events_region ON events(region)")
print("after:",conn.execute("EXPLAIN QUERY PLAN "+query).fetchall())
assert len(conn.execute(query).fetchall()) == 2

before: [(2, 0, 0, 'SCAN events')]
after: [(3, 0, 0, 'SEARCH events USING INDEX idx_events_region (region=?)')]


## 4. תרגיל שדורש מחשבה
השלם שאילתת `GROUP BY` שמחזירה **מספר אירועים ייחודיים בכל אזור**, גם כשלטבלת `tags` יש 0–N תגיות לאירוע. כתוב מדוע `COUNT(DISTINCT e.event_id)` שונה מ־`COUNT(*)`.

**Failure injection:** הוסף שתי תגיות חדשות ל־e2; בדוק אילו queries השתנו ואילו אמורות להישאר יציבות.

**Next:** [Stage 01](../curriculum/modules/01.md) → [Notebook 02](02_etl_failure_lab.ipynb).